# Can we predict whether a student passes the closing quiz?
Target: `passed_quiz`. Follows `checklist.md`.

**Design decisions (made before modelling)**
- No timestamp column exists, so there is no "later rows" split. `student_key` repeats (about 7 rows per student), so every split (test set and CV folds) keeps whole students together.
- Prediction time: just before the closing quiz is taken. `prior_score` (placement test), `enrol_year`, `school_code`, `session_minutes` and `hints_used` are all recorded by then. `student_key` is an identifier and is dropped.
- Model choice uses ROC AUC; the decision threshold is chosen on out-of-fold CV predictions to maximise accuracy.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, accuracy_score
df = pd.read_csv('data.csv')
print(df.shape, df.isna().sum().sum(), 'missing;', df.student_key.nunique(), 'students; positive rate', df.passed_quiz.mean().round(3))

## 1. Hold out a test set (whole students) before any modelling

In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
tr_idx, te_idx = next(gss.split(df, groups=df.student_key))
train, test = df.iloc[tr_idx].reset_index(drop=True), df.iloc[te_idx].reset_index(drop=True)
assert not set(train.student_key) & set(test.student_key)
print(len(train), len(test))
# the test set is not touched again until step 6

## 4. Candidate models (named before any scoring)
1. `baseline`: predicts the training class prior.
2. `logreg_session`: standardised logistic regression on `prior_score, session_minutes, hints_used`.
3. `logreg_all`: same plus `enrol_year, school_code` (as numbers).
4. `hgb_session`: gradient boosting on the three session features.
5. `hgb_all`: gradient boosting on all five features.

Hyperparameters are fixed in advance (logreg C=1; HGB max_depth=3, learning_rate=0.05, max_iter=200, l2=1.0), so the only selection is among these five.

In [ ]:
S = ['prior_score', 'session_minutes', 'hints_used']; A = S + ['enrol_year', 'school_code']
hgb = lambda: HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, l2_regularization=1.0, random_state=0)
candidates = {
    'baseline':       (lambda: DummyClassifier(strategy='prior'), S),
    'logreg_session': (lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0)), S),
    'logreg_all':     (lambda: make_pipeline(StandardScaler(), LogisticRegression(C=1.0)), A),
    'hgb_session':    (hgb, S),
    'hgb_all':        (hgb, A),
}

## 2-3. Grouped cross-validation on training rows only
Scaling lives inside the pipeline, so it is fit within each fold. Out-of-fold predictions are kept for the threshold choice.

In [ ]:
folds = list(GroupKFold(n_splits=5).split(train, groups=train.student_key))
y = train.passed_quiz.values
oof, rows = {}, []
for name, (mk, feats) in candidates.items():
    p = np.zeros(len(train)); aucs = []
    for a, b in folds:
        m = mk().fit(train.iloc[a][feats], y[a])
        p[b] = m.predict_proba(train.iloc[b][feats])[:, 1]
        aucs.append(roc_auc_score(y[b], p[b]))
    oof[name] = p; rows.append((name, np.mean(aucs), np.std(aucs)))
cv = pd.DataFrame(rows, columns=['model', 'cv_auc_mean', 'cv_auc_sd']).set_index('model')
cv

## 5. Choose model and threshold on CV only

In [ ]:
best = cv.cv_auc_mean.idxmax()
# prefer the simpler (earlier-listed) model if within one SD of the best
for name in ['logreg_session', 'logreg_all', 'hgb_session', 'hgb_all']:
    if cv.loc[name, 'cv_auc_mean'] >= cv.loc[best, 'cv_auc_mean'] - cv.loc[best, 'cv_auc_sd']:
        best = name; break
ths = np.linspace(0.2, 0.8, 61)
accs = [accuracy_score(y, oof[best] > t) for t in ths]
thr = float(ths[int(np.argmax(accs))])
print('chosen model:', best, '| threshold:', round(thr, 2), '| CV accuracy at threshold:', round(max(accs), 3),
      '| majority-class accuracy:', round(max(y.mean(), 1 - y.mean()), 3))

**Choice recorded here, before test scoring:** the model and threshold printed above are final. The test set is scored once, next.

## 6. Refit on all training rows; score the test set once

In [ ]:
mk, feats = candidates[best]
final = mk().fit(train[feats], y)
pt = final.predict_proba(test[feats])[:, 1]; yt = test.passed_quiz.values
print('TEST AUC      :', round(roc_auc_score(yt, pt), 3))
print('TEST accuracy :', round(accuracy_score(yt, pt > thr), 3), '(majority-class accuracy', round(max(yt.mean(), 1 - yt.mean()), 3), ')')
# uncertainty: bootstrap over test students
rng = np.random.default_rng(0); g = test.student_key.values; ids = np.unique(g)
idx = {s: np.where(g == s)[0] for s in ids}; bs = []
for _ in range(500):
    ii = np.concatenate([idx[s] for s in rng.choice(ids, len(ids))])
    bs.append(roc_auc_score(yt[ii], pt[ii]))
print('95% bootstrap CI for AUC:', np.percentile(bs, [2.5, 97.5]).round(3))

## Conclusion
The headline is the single test-set score from step 6, with its bootstrap interval. Read AUC against 0.5 (no skill) and accuracy against the majority-class rate. Caveats: there is no time column, so we test generalisation to unseen students, not to future sessions; `prior_score` has values above 100, which we left as is.